# Week 3 · Try It: Detect NLOS Satellites with Logistic Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%203/logistic_nlos.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

In a city canyon a GNSS receiver often tracks a signal reflected by a building (NLOS). NLOS signals tend to have a lower carrier-to-noise ratio (SNR, dB-Hz) and a lower elevation. The 400 satellites below are simulated; label 1 = NLOS, 0 = line of sight (LOS).

**Tasks**
1. Train a logistic-regression classifier on SNR and elevation (Part 1).
2. Compute P(NLOS) for satellites A and B from the slide *UAS Case: Detecting NLOS GNSS Signals* (Part 2).
3. Move the decision threshold from 0.5 to 0.3 and watch precision and recall (Part 3).

## Part 1 · Data and training

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(4011)
n_los, n_nlos = 240, 160
snr = np.r_[rng.normal(42, 5, n_los), rng.normal(32, 5, n_nlos)]          # dB-Hz
elev = np.clip(np.r_[rng.normal(45, 15, n_los), rng.normal(22, 12, n_nlos)], 5, 88)  # degrees
X = np.column_stack([snr, elev])
y = np.r_[np.zeros(n_los, int), np.ones(n_nlos, int)]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression().fit(X_tr, y_tr)
th0, (th1, th2) = clf.intercept_[0], clf.coef_[0]
print(f"P(NLOS) = sigma({th0:.2f} {th1:+.3f}*SNR {th2:+.3f}*elev)")

**Question.** Why are both weights negative?

## Part 2 · Two satellites

In [ ]:
for name, s, e in (("A", 45, 60), ("B", 25, 15)):
    p = clf.predict_proba([[s, e]])[0, 1]
    print(f"satellite {name}: SNR {s} dB-Hz, elevation {e} deg -> P(NLOS) = {p:.3f}")

## Part 3 · Moving the threshold

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

p_te = clf.predict_proba(X_te)[:, 1]
for thr in (0.5, 0.3):                  # try 0.7 and 0.1 as well
    pred = (p_te >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    print(f"threshold {thr}: TP {tp}, FP {fp}, FN {fn}, TN {tn},  "
          f"precision {precision_score(y_te, pred):.3f}, recall {recall_score(y_te, pred):.3f}")

**Questions.**
1. What happens to precision and recall when the threshold goes down? Why?
2. For positioning safety, which error is worse: a missed NLOS signal (FN) or a good signal thrown away (FP)? Which threshold would you choose?